In [ ]:
-- Анализ стабильности номиналов между последовательными кампаниями
-- Проверяем, как часто клиент получает тот же номинал бонусов
-- в следующем флайте механики УВ

with prepared as (

    -- Подготовка данных:
    -- извлекаем числовой номинал бонусов из текстового описания сегмента

    select
        client_id,
        campaign_name,
        mrc_start_date,

        case
            when lower(com_cus_sgr_desc) like '%200%' then 200
            when lower(com_cus_sgr_desc) like '%300%' then 300
            when lower(com_cus_sgr_desc) like '%400%' then 400
            when lower(com_cus_sgr_desc) like '%500%' then 500
            else null
        end as nominal_amount

    from YOUR_TABLE

    where client_id is not null
),

client_nominal_history as (

    -- Для каждого клиента:
    -- сортируем кампании по времени
    -- и получаем предыдущий номинал бонусов

    select
        client_id,
        campaign_name,
        mrc_start_date,
        nominal_amount,

        lag(nominal_amount) over (
            partition by client_id
            order by mrc_start_date
        ) as prev_nominal

    from prepared

    where nominal_amount is not null
),

nominal_comparison as (

    -- Сравниваем текущий номинал с предыдущим
    -- 1 = номинал не изменился
    -- 0 = номинал изменился

    select
        *,
        case
            when nominal_amount = prev_nominal then 1
            else 0
        end as same_nominal_flag

    from client_nominal_history

    -- Исключаем первую кампанию клиента,
    -- так как для нее нет предыдущего номинала

    where prev_nominal is not null
)

-- Финальная агрегация:
-- считаем долю переходов,
-- в которых номинал сохранился

select
    count(*) as transitions_cnt,

    sum(same_nominal_flag) as same_nominal_cnt,

    round(
        sum(same_nominal_flag) * 100.0 /
        count(*),
        2
    ) as same_nominal_pct

from nominal_comparison;